# Archived experiment notebook

Source preserved for review. Private outputs, attachments, and execution counts are removed in this public copy; original AWS notebooks remain untouched. This copy is not evidence of a fresh execution. Use the public Research Review for aggregate results. Private input contracts and artifacts are required to reproduce this historical experiment.


# 05 · Feature attribution and chronological screening
## NFL trajectory research · Round 1

**This is a small diagnostic, not the saved neural model and not a Kaggle score.** It answers whether the candidate information helps a fixed inexpensive interface, and whether the reconstructed views are fit-ready. It cannot rule out a feature or augmentation that needs a temporal neural encoder.

| Arm | Inputs | Training policy |
|---|---|---|
| `control` | Existing-domain control representation | Original origins only |
| `arrival` | Control plus role-conditioned arrival features | Same original training rows |
| `arrival_origin` | Same input width as `arrival` | Half original, half earlier-origin rows |

**Primary contrasts:** arrival minus control; arrival_origin minus arrival. No ensemble, hyperparameter search, test-score tuning or reuse of a fitted parent baseline. Each arm has the same labeled training-row budget within its fold; only the third changes which training views occupy that budget.

Chronological folds are inside the original training partition. Validation is always at origin zero, with every requested row of each selected evaluation play retained. Features are centered/scaled and constant columns screened on each arm's training rows only.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import numpy as np
import pandas as pd

KIT = Path.cwd()
if not (KIT / "run_round.py").is_file():
    KIT = Path("/home/sagemaker-user/nfl_feature_round1")
if not (KIT / "run_round.py").is_file():
    raise FileNotFoundError("Open this notebook from the extracted nfl_feature_round1 folder.")
sys.path.insert(0, str(KIT))
from visuals import show_save
OUT = Path("/home/sagemaker-user/nfl-feature-round1-results")
FIGURES = OUT / "figures"

def run_stage(command, *arguments, seconds=600):
    cmd = [sys.executable, str(KIT / "run_round.py"), command, *map(str, arguments), "--seconds", str(seconds)]
    print("Running:", " ".join(cmd), flush=True)
    subprocess.run(cmd, cwd=KIT, check=True, timeout=seconds + 30)

print("Kernel:", sys.executable)
print("Outputs:", OUT)


## 1 · Run nine small, checkpointed fits
The hard cap is **600 seconds**. Fits use a fixed ridge penalty of 0.01, at most 10,000 training rows per arm, and two numerical-library threads. No expensive established model is retrained. The exact fit can be restored from saved numeric arrays and replayed; this is tested after every new fit.

Repeating this stage with unchanged inputs reuses its nine fitted artifacts. A changed source/environment/protocol is rejected rather than silently reusing incompatible results.

In [ ]:
prepared = json.loads((OUT / "research" / "preparation_summary.json").read_text())
assert prepared["status"] == "feature_dataset_ready" and prepared["selected_training_plays"] >= 128
run_stage("screen", "--label", "research", seconds=600)
result = json.loads((OUT / "research" / "screen_summary.json").read_text())
assert result["status"] == "diagnostic_screen_complete"
assert not result["outer_validation_used"] and result["kaggle_score"] is None
print(result["scope"])
display(pd.DataFrame(result["fits"]))

## 2 · Pooled official-form metric
The mathematical metric is `sqrt(sum(dx² + dy²) / (2N))`, in yards. Pool squared errors and row counts; do not average fold RMSEs. Dataset, model and comparison scope still differ from the leaderboard.

In [ ]:
from visuals import pooled_figure, folds_figure
show_save(pooled_figure(result), FIGURES / "pooled_feature_screen.html")
show_save(folds_figure(result), FIGURES / "chronological_feature_screen.html")

## 3 · Contribution and uncertainty
Negative treatment-minus-control deltas favor the treatment. Paired resampling keeps both arms' predictions aligned and resamples games rather than individual frames. Intervals are conditional on these fitted models; they do not capture training-seed uncertainty.

The predeclared **exploratory** continuation gate requires at least 1% lower pooled RMSE, a negative upper 95% paired-game interval, and improvement in at least two of three folds. There are two contrasts; intervals are not multiplicity-adjusted or confirmatory. A pass earns stronger-model integration, not deployment or a claim that feature research is complete.

In [ ]:
from visuals import intervals_figure
show_save(intervals_figure(result), FIGURES / "paired_feature_intervals.html")
display(pd.DataFrame(result["decisions"]))

## 4 · Where the difference occurs
Long-horizon improvement is a diagnostic, not permission to select weights or feature gates after seeing results. No late rows are dropped and no horizon is clamped to make the score look better.

In [ ]:
from visuals import slice_figure
show_save(slice_figure(result), FIGURES / "horizon_feature_diagnostics.html")
display(pd.DataFrame(result["slices"]))

## 5 · Save the decision and export aggregate evidence
A negative result stops this specific ridge treatment, not the broader domain hypothesis. A positive result must next be checked in the established neural representation under a matched compute budget, then replicated across folds/seeds. Do not launch a large GPU job from this notebook.

The return ZIP contains aggregate receipts only. Private row-level features, labels, errors and weights stay in the results directory. The new research notebooks are not automatically published or committed.

In [ ]:
review = {
    "attempted": "24 role-conditioned arrival candidates and earlier-origin training, fixed-ridge diagnostic",
    "completed": result["status"],
    "passed": {"exact_forward_replay_all_fits": all(r["forward_replay_exact"] for r in result["fits"]),
               "outer_validation_not_used": not result["outer_validation_used"]},
    "actual_metric": result["pooled_rmse"],
    "decisions": result["decisions"],
    "new_fits": result["new_fits"], "reused_fits": result["reused_fits"],
    "github_updated": False, "kaggle_submitted": False,
    "next": "Review the report before selecting the next feature/augmentation integration experiment."
}
from run_round import atomic_json
atomic_json(OUT / "research" / "milestone_review.json", review)
display(pd.Series(review))
run_stage("report", seconds=60)
print("Download and attach:", OUT / "nfl_feature_round1_report.zip")